# C-MET 科学小规模复现：从“能跑通”到“可验证”

这个 Notebook 在几小时内完成一条**推理级科学复现闭环**。它不仅生成视频，还固定版本、记录环境与权重哈希，并执行控制组、样本数消融和随机种子敏感性实验。

默认 `scientific` 配置生成 6 个视频：3 个主实验、1 个零情绪方向控制组、1 个单样本消融、1 个随机种子敏感性实验。A100/L4 上通常约 2～6 小时。

> 这不是 20 万 step 训练复现，也不声称复现作者未公开的精确 FID/FVD/SyncNet/Emotion-FAN 协议。


## 使用方法

1. 在 Colab 菜单中选择 GPU 运行时。
2. 直接运行下一格“一键科学小规模复现”。
3. Google Drive 授权时点击允许。
4. 等待输出 `科学小规模复现完成`。
5. Notebook 会展示自动报告和全部生成视频。

默认参数无需修改。结果永久保存到 `MyDrive/C-MET-mini/results/scientific_mini/`。断线后重新运行同一格，已完成且通过检查的实验会自动跳过。

`PROFILE="demo"` 只生成主情绪，约 1～3 小时；`PROFILE="scientific"` 加入对照、消融和敏感性分析，约 2～6 小时。


In [ ]:
# @title ▶ 一键科学小规模复现（默认约 2～6 小时）
PROFILE = "scientific" # @param ["scientific", "demo"]
EMOTIONS = "happy,sad,angry" # @param {type:"string"}
NUM_SAMPLES = 3 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
SENSITIVITY_SEED = 123 # @param {type:"integer"}
RUN_NAME = "scientific_mini" # @param {type:"string"}

import shlex
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
from IPython.display import Markdown, Video, display


def run(command, *, cwd=None):
    command = [str(value) for value in command]
    print("$", shlex.join(command), flush=True)
    subprocess.run(command, cwd=str(cwd) if cwd else None, check=True)


if NUM_SAMPLES < 1 or NUM_SAMPLES > 10:
    raise ValueError("NUM_SAMPLES 必须保持在 1～10。")
if PROFILE not in {"scientific", "demo"}:
    raise ValueError("PROFILE 只能是 scientific 或 demo。")

print("== 0/6 检查 GPU ==", flush=True)
try:
    import torch
except Exception as exc:
    raise RuntimeError("无法导入 Colab 自带的 PyTorch。") from exc
if not torch.cuda.is_available():
    raise RuntimeError("没有检测到 GPU。请在‘运行时 → 更改运行时类型’中选择 GPU。")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
subprocess.run(["nvidia-smi"], check=False)

print("== 1/6 挂载 Google Drive ==", flush=True)
DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)
MY_DRIVE = DRIVE_MOUNT / "MyDrive"
if not MY_DRIVE.is_dir():
    raise RuntimeError("Google Drive 没有正确挂载。")

print("== 2/6 克隆或更新 mini 项目 ==", flush=True)
PROJECT_ROOT = Path("/content/cmet-mini-repro-colab")
PROJECT_URL = "https://github.com/ChengyangHe-ux/cmet-repro-colab.git"
PROJECT_BRANCH = "mini-repro"
if not PROJECT_ROOT.exists():
    run(["git", "clone", "--branch", PROJECT_BRANCH, "--single-branch", PROJECT_URL, PROJECT_ROOT])
elif not (PROJECT_ROOT / ".git").is_dir():
    raise RuntimeError(f"路径已存在但不是 Git 仓库：{PROJECT_ROOT}")
else:
    run(["git", "-C", PROJECT_ROOT, "pull", "--ff-only"])

print("== 3/6 安装系统依赖 ==", flush=True)
missing = [name for name in ["ffmpeg", "ffprobe", "git-lfs"] if shutil.which(name) is None]
if missing:
    run(["apt-get", "update", "-y"])
    run(["apt-get", "install", "-y", "ffmpeg", "git-lfs"])
else:
    print("系统依赖已经存在，跳过 apt。", flush=True)

print("== 4/6 安装 Python 依赖 ==", flush=True)
run(
    [
        sys.executable,
        PROJECT_ROOT / "scripts" / "install_colab_dependencies.py",
        "--requirements",
        PROJECT_ROOT / "configs" / "colab_requirements.txt",
    ]
)

print("== 5/6 运行官方 checkpoint 科学小规模复现 ==", flush=True)
DRIVE_ROOT = MY_DRIVE / "C-MET-mini"
run(
    [
        sys.executable,
        PROJECT_ROOT / "scripts" / "run_colab_mini.py",
        "--project-root",
        PROJECT_ROOT,
        "--drive-root",
        DRIVE_ROOT,
        "--profile",
        PROFILE,
        "--emotions",
        EMOTIONS,
        "--num-samples",
        NUM_SAMPLES,
        "--seed",
        SEED,
        "--sensitivity-seed",
        SENSITIVITY_SEED,
        "--run-name",
        RUN_NAME,
    ]
)

print("== 6/6 展示科学报告和结果 ==", flush=True)
OUTPUT_ROOT = DRIVE_ROOT / "results" / RUN_NAME
REPORT_PATH = OUTPUT_ROOT / "report.md"
if REPORT_PATH.is_file():
    display(Markdown(REPORT_PATH.read_text(encoding="utf-8")))
else:
    raise RuntimeError(f"没有找到自动报告：{REPORT_PATH}")

videos = sorted(OUTPUT_ROOT.glob("*.mp4"))
if not videos:
    raise RuntimeError(f"没有找到生成视频：{OUTPUT_ROOT}")
for video in videos:
    print("结果：", video.name)
    display(Video(str(video), embed=True, width=512))
print("完成。报告、JSON 记录和视频已保存到：", OUTPUT_ROOT)


## 这条科学小规模链路验证了什么

- 官方 C-MET 代码、官方 checkpoint 和示例素材能否在固定环境中组成完整推理链路；
- 目标情绪方向相对“零方向控制组”是否导致可记录的输出变化；
- 情绪特征样本数从 3 改为 1 后，方向范数和视频输出是否变化；
- 改变特征抽样种子后，结果是否仍能稳定生成；
- 每个结果是否有有效时长，并同时包含视频流和音频流；
- 环境、权重、输入、参数、耗时和分析结果是否可以完整追溯。

自动报告中的像素 MAD 只能证明输出发生变化，不能证明情绪语义一定正确。情绪准确性仍需要人工盲评或独立分类器。
